## Setup

### General

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd

from _utilities.data_split import make_split, set_seeds
from _utilities.stupidloggen import StupidLogGenerator


### Paths & Logger

In [2]:
ROOT_DIR = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(ROOT_DIR)

PATH_ORG_DATA = "data/compas-scores-two-years.csv"
PATH_SYNTHETIC_DATA = "data/synthetic/"
PATH_FOLDER_MODELS = "models/"

if not os.path.exists(PATH_FOLDER_MODELS):
    os.makedirs(PATH_FOLDER_MODELS)
if not os.path.exists(PATH_SYNTHETIC_DATA):
    os.makedirs(PATH_SYNTHETIC_DATA)

In [3]:
logger = StupidLogGenerator().initLogger()

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- is_recid

In [4]:
scores_two_years = pd.read_csv(PATH_ORG_DATA)

columns_to_include = [
    "sex",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
    "is_recid",
]

dataset = scores_two_years[columns_to_include].copy()

In [5]:
categorical_columns = [
    "sex",
    "age_cat",
    "race",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
    "is_recid",
]

In [6]:
dataset.nunique()

sex                  2
age                 65
age_cat              3
race                 6
juv_fel_count       11
juv_misd_count      10
juv_other_count     10
priors_count        37
c_charge_degree      2
c_charge_desc      437
r_charge_degree     10
r_charge_desc      340
two_year_recid       2
is_recid             2
dtype: int64

## Data Preprocessing

When `is_recid` is 0 there should not be any values in `r_charge_degree` and `r_charge_desc`, as they contain information about the crime they were re-arested for. Thus, we made a separate category "Not applicable" to make it clear which values are truly missing.

In [7]:
recid_charge_cols = [
    "r_charge_degree",
    "r_charge_desc",
]

for col in recid_charge_cols:
    dataset[col] = np.where((dataset["is_recid"] == 0), "Not applicable", dataset[col])

The `race` attribute is distributed as indicated in the table below. The ammount of data on "Hispanic", "Asian", and "Native American" individually is not enough to be significant, therefore we combine these together with "Other".

| Race             | Count | Percent |
| --- | --- | ---  |
| African-American |  3696 |  51.23% |
| Caucasian        |  2454 |  34.02% |
| Hispanic         |   637 |   8.83% |
| Other            |   377 |   5.23% |
| Asian            |    32 |   0.44% |
| Native American  |    18 |   0.25% |

In [8]:
other_races = ["Hispanic", "Asian", "Native American"]

dataset["race"] = dataset["race"].replace(other_races, "Other")

In [9]:
dataset.to_csv("data/preprocessed_real_data.csv", index=False)

In [10]:
train_df, test_df = make_split(dataset)

train: 5771 rows | test: 1443 rows
reoffending rate by sex (train):
sex
Female    0.384
Male      0.514
Name: is_recid, dtype: float64


## Model Training

#### Model Setup

In [11]:
# To make sure that FairGAN can be imported
path_to_dir = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(path_to_dir)

In [12]:
import pickle

from ctgan import CTGAN, TVAE
from tabfairgan import TFG

from fairgan import FairGAN


In [13]:
def save_successful(model, model_name: str, synthetic_data: pd.DataFrame):
    try:
        synthetic_data.to_csv(
            PATH_SYNTHETIC_DATA + f"{model_name}_synthetic.csv", index=False
        )

        with open(PATH_FOLDER_MODELS + f"{model_name}_model.pkl", "wb") as f:
            pickle.dump(model, f)
    except Exception as e:
        logger.error(e)

#### CTGAN

The CTGAN model learns the joint distribution of the dataset.

In [14]:
def run_CTGAN(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    verbose: bool,
    synthetic_data_count: int,
):
    logger.info("\n\nStarting to train CTGAN model")

    try:
        # Train CTGAN model
        ctgan_df = df.copy()

        set_seeds()
        ctgan_model = CTGAN(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
        ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

        # Generate synthetic data using CTGAN
        ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)

        save_successful(ctgan_model, "ctgan", ctgan_synthetic_data)

    except Exception as e:
        logger.error(e)

#### TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [15]:
def run_TVAE(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    verbose: bool,
    synthetic_data_count: int,
):
    logger.info("\n\nStarting to train TVAE model")

    try:
        # Train TVAE model
        tvae_df = df.copy()

        set_seeds()

        tvae_model = TVAE(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
        tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

        # Generate synthetic data using TVAE
        tvae_synthetic_data = tvae_model.sample(synthetic_data_count)

        save_successful(tvae_model, "tvae", tvae_synthetic_data)
    except Exception as e:
        logger.error(e)

#### FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

In [16]:
def run_FairGAN(
    df: pd.DataFrame,
    sensitive_feature: str,
    target_feature: str,
    pretrain_epochs: int,
    epochs: int,
    batch_size: int,
    noise_dim: int,
    lr: float,
    synthetic_data_count: int,
    verbose: bool,
):
    logger.info("\n\nStarting to train FairGAN with different lambda_fair values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    fairgan_datasets = {}
    fairgan_models = {}

    for lam in lambda_values:
        logger.info(f"\nTraining FairGAN with lambda_fair = {lam}")

        fairgan_df = df.copy()

        try:
            set_seeds()

            # Initialize model
            fairgan_model = FairGAN(
                df=fairgan_df,
                sensitive_feature=sensitive_feature,
                target_feature=target_feature,
                pretrain_epochs=pretrain_epochs,
                epochs=epochs,
                batch_size=batch_size,
                noise_dim=noise_dim,
                lr=lr,
                verbose=verbose,
            )

            # Train model
            fairgan_model.train(lambda_fair=lam)

            # Generate synthetic dataset
            synthetic_df = fairgan_model.generate(synthetic_data_count)

            dataset_name = f"FairGAN_lambda_{lam}"
            fairgan_datasets[dataset_name] = synthetic_df
            fairgan_models[dataset_name] = fairgan_model

            save_successful(fairgan_model, dataset_name, synthetic_df)

            logger.info(f"Completed {dataset_name}")

        except Exception as e:
            logger.error(e)

#### TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

**Fairness Constraints**

To generate fair synthetic data, we need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [17]:
def run_TabFairGAN(
    df: pd.DataFrame,
    num_epochs: int,
    batch_size: int,
    synthetic_data_count: int,
    fairness_config: dict,
):
    logger.info("\n\nStarting to train TabFairGAN with different lambda values")

    lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

    tabfairgan_datasets = {}
    tabfairgan_models = {}

    for lam in lambda_values:
        if lam == 0.0:
            fairness_config["fair_epochs"] = 0
        else:
            fairness_config["fair_epochs"] = 50

        fairness_config["lamda"] = lam

        try:
            logger.info(f"\nTraining TabFairGAN with lambda = {lam}")

            tfg_df = df.copy()

            set_seeds()

            # Turn the sensitive and target attribute into categorical string
            tfg_df["sex"] = tfg_df["sex"].astype(str)
            tfg_df["is_recid"] = tfg_df["is_recid"].astype(str)

            # Initialize model
            tfg_model = TFG(
                tfg_df,
                epochs=num_epochs,
                batch_size=batch_size,
                fairness_config=fairness_config,
            )

            # Train model
            tfg_model.train()

            # Generate synthetic data
            tfg_synthetic_data = tfg_model.generate_fake_df(
                num_rows=synthetic_data_count
            )

            dataset_name = f"TabFairGAN_lambda_{lam}"
            tabfairgan_datasets[dataset_name] = tfg_synthetic_data
            tabfairgan_models[dataset_name] = tfg_model

            save_successful(tfg_model, dataset_name, tfg_synthetic_data)

            logger.info(f"Completed {dataset_name}")

        except Exception as e:
            logger.error(e)
            continue

## Model Settings

In [18]:
# Set logging level
logger.setLevel(logging.INFO)

In [19]:
target_feature = "is_recid"
sensitive_feature = "sex"
sensitive_under = "Female"

fairness_config = {
    "S": sensitive_feature,
    "Y": target_feature,
    "S_under": sensitive_under,
    "Y_desire": "0",
}

In [20]:
verbose = 2
synthetic_data_count = 5000
num_epochs = 200
batch_size = 120

# FairGAN hyperparameters
num_pretrain_epochs = 20
fairgan_noise_dim = 100
fairgan_lr = 0.001

## Run models

In [21]:
run_CTGAN(dataset, num_epochs, batch_size, verbose, synthetic_data_count)
run_TVAE(dataset, num_epochs, batch_size, verbose, synthetic_data_count)

run_FairGAN(
    dataset,
    sensitive_feature,
    target_feature,
    pretrain_epochs=num_pretrain_epochs,
    epochs=num_epochs,
    batch_size=batch_size,
    noise_dim=fairgan_noise_dim,
    lr=fairgan_lr,
    synthetic_data_count=synthetic_data_count,
    verbose=verbose,
)

run_TabFairGAN(dataset, num_epochs, batch_size, synthetic_data_count, fairness_config)


[INFO]: 

Starting to train CTGAN model
Gen. (+00.52) | Discrim. (-00.11): 100%|██████████| 200/200 [28:47<00:00,  8.64s/it]
[INFO]: 

Starting to train TVAE model
Loss: -07.67: 100%|██████████| 200/200 [02:24<00:00,  1.38it/s]
[INFO]: 

Starting to train FairGAN with different lambda_fair values
[INFO]: 
Training FairGAN with lambda_fair = 0.0


[Pretrain] epoch 0 | loss 8394.2324
[Pretrain] epoch 5 | loss 5744.9785
[Pretrain] epoch 10 | loss 2893.2769
[Pretrain] epoch 15 | loss 747.1729
Epoch 0 | D1: 0.8717 | D2: 1.3723 | G: 0.9179 | Fair: 0.6940 | y_hat mean: 0.288 | y_hat std: 0.087
Epoch 10 | D1: 0.7065 | D2: 1.0899 | G: 1.7732 | Fair: 0.7629 | y_hat mean: 0.004 | y_hat std: 0.004
Epoch 20 | D1: 0.8142 | D2: 1.0222 | G: 1.8799 | Fair: 1.1698 | y_hat mean: 0.998 | y_hat std: 0.003
Epoch 30 | D1: 1.2208 | D2: 1.0856 | G: 1.0091 | Fair: 1.2484 | y_hat mean: 0.994 | y_hat std: 0.005
Epoch 40 | D1: 1.5233 | D2: 1.5684 | G: 1.0283 | Fair: 0.9084 | y_hat mean: 0.616 | y_hat std: 0.426
Epoch 50 | D1: 1.2931 | D2: 2.3132 | G: 1.3506 | Fair: 0.8305 | y_hat mean: 0.652 | y_hat std: 0.412
Epoch 60 | D1: 1.0334 | D2: 1.1600 | G: 1.6993 | Fair: 0.7514 | y_hat mean: 0.048 | y_hat std: 0.121
Epoch 70 | D1: 1.4509 | D2: 1.4661 | G: 0.9333 | Fair: 0.7373 | y_hat mean: 0.029 | y_hat std: 0.061
Epoch 80 | D1: 1.3843 | D2: 1.3619 | G: 0.9272 |

[INFO]: Completed FairGAN_lambda_0.0
[INFO]: 
Training FairGAN with lambda_fair = 0.1


[Pretrain] epoch 0 | loss 8394.2324
[Pretrain] epoch 5 | loss 5744.9785
[Pretrain] epoch 10 | loss 2893.2769
[Pretrain] epoch 15 | loss 747.1729
Epoch 0 | D1: 0.8708 | D2: 1.3723 | G: 0.9200 | Fair: 0.6940 | y_hat mean: 0.288 | y_hat std: 0.087
Epoch 10 | D1: 0.7002 | D2: 0.8954 | G: 1.9176 | Fair: 0.7607 | y_hat mean: 0.007 | y_hat std: 0.010
Epoch 20 | D1: 0.8980 | D2: 0.9906 | G: 2.1334 | Fair: 1.3238 | y_hat mean: 0.549 | y_hat std: 0.302
Epoch 30 | D1: 1.1031 | D2: 0.6871 | G: 0.9340 | Fair: 1.0005 | y_hat mean: 0.997 | y_hat std: 0.002
Epoch 40 | D1: 0.9696 | D2: 1.5672 | G: 1.1585 | Fair: 0.8439 | y_hat mean: 0.003 | y_hat std: 0.004
Epoch 50 | D1: 0.8828 | D2: 1.7337 | G: 2.5129 | Fair: 0.9417 | y_hat mean: 0.966 | y_hat std: 0.072
Epoch 60 | D1: 1.6275 | D2: 0.9487 | G: 1.3379 | Fair: 0.9824 | y_hat mean: 0.995 | y_hat std: 0.005
Epoch 70 | D1: 1.1440 | D2: 0.9792 | G: 1.5764 | Fair: 0.7431 | y_hat mean: 0.206 | y_hat std: 0.262
Epoch 80 | D1: 1.7036 | D2: 1.4852 | G: 1.2654 |

[INFO]: Completed FairGAN_lambda_0.1
[INFO]: 
Training FairGAN with lambda_fair = 0.5


...

epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10
epoch 11
epoch 12
epoch 13
epoch 14
epoch 15
epoch 16
epoch 17
epoch 18
epoch 19
epoch 20
epoch 21
epoch 22
epoch 23
epoch 24
epoch 25
epoch 26
epoch 27
epoch 28
epoch 29
epoch 30
epoch 31
epoch 32
epoch 33
epoch 34
epoch 35
epoch 36
epoch 37
epoch 38
epoch 39
epoch 40
epoch 41
epoch 42
epoch 43
epoch 44
epoch 45
epoch 46
epoch 47
epoch 48
epoch 49
epoch 50
epoch 51
epoch 52
epoch 53
epoch 54
epoch 55
epoch 56
epoch 57
epoch 58
epoch 59
epoch 60
epoch 61
epoch 62
epoch 63
epoch 64
epoch 65
epoch 66
epoch 67
epoch 68
epoch 69
epoch 70
epoch 71
epoch 72
epoch 73
epoch 74
epoch 75
epoch 76
epoch 77
epoch 78
epoch 79
epoch 80
epoch 81
epoch 82
epoch 83
epoch 84
epoch 85
epoch 86
epoch 87
epoch 88
epoch 89
epoch 90
epoch 91
epoch 92
epoch 93
epoch 94
epoch 95
epoch 96
epoch 97
epoch 98
epoch 99
epoch 100
epoch 101
epoch 102
epoch 103
epoch 104
epoch 105
epoch 106
epoch 107
epoch 108
epoch 109
epoch 110
epoch 11

[INFO]: Completed TabFairGAN_lambda_0.7
[INFO]: 
Training TabFairGAN with lambda = 1.0


Training Progress:   0%|                                                    | 0/200 [00:00<?, ?it/s]

epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10
epoch 11
epoch 12
epoch 13
epoch 14
epoch 15
epoch 16
epoch 17
epoch 18
epoch 19
epoch 20
epoch 21
epoch 22
epoch 23
epoch 24
epoch 25
epoch 26
epoch 27
epoch 28
epoch 29
epoch 30
epoch 31
epoch 32
epoch 33
epoch 34
epoch 35
epoch 36
epoch 37
epoch 38
epoch 39
epoch 40
epoch 41
epoch 42
epoch 43
epoch 44
epoch 45
epoch 46
epoch 47
epoch 48
epoch 49
epoch 50
epoch 51
epoch 52
epoch 53
epoch 54
epoch 55
epoch 56
epoch 57
epoch 58
epoch 59
epoch 60
epoch 61
epoch 62
epoch 63
epoch 64
epoch 65
epoch 66
epoch 67
epoch 68
epoch 69
epoch 70
epoch 71
epoch 72
epoch 73
epoch 74
epoch 75
epoch 76
epoch 77
epoch 78
epoch 79
epoch 80
epoch 81
epoch 82
epoch 83
epoch 84
epoch 85
epoch 86
epoch 87
epoch 88
epoch 89
epoch 90
epoch 91
epoch 92
epoch 93
epoch 94
epoch 95
epoch 96
epoch 97
epoch 98
epoch 99
epoch 100
epoch 101
epoch 102
epoch 103
epoch 104
epoch 105
epoch 106
epoch 107
epoch 108
epoch 109
epoch 110
epoch 11

[INFO]: Completed TabFairGAN_lambda_1.0
